Filtrer ut støy ved å si at alle linjer som ikke begynner med '[' er støy.
Videre kan hver linje legges til i ei liste hvor elementene per linje er adskilt med 'mellomrom'


TASK 1

In [13]:

from pathlib import Path
import json
import numpy as np
import math as m
from typing import Any
import pandas as pd

In [14]:

def find_encoding(file: Path):
    """Checks if encoding of file is UTF-8, if not, sets encoding to Latin 1"""
    file = Path(file)
    try:
        with file.open(encoding= 'utf-8') as f:
            f.read()
        Etype = 'utf-8'
    
    except UnicodeDecodeError:
        Etype = 'iso-8859-1'

    return Etype

        
# I had to alter one of the elements in the 'Cgenerate_data_d2.py' because the vreated files only contained ASCII chars,
#so it wasnt possible to separate utf-8 from latin 1 without æ,ø,å. To solve, 'muffin' was changed to 'møffin'

from pathlib import Path

# Dette henter mappen der notatblokken din ligger, automatisk!
BASE_DIR = Path().resolve() 

# Bruk BASE_DIR for å peke helt nøyaktig uansett hvor kernelen kjører:
a = find_encoding(BASE_DIR/ '2026/library_cafe/POS_01/register_sales.log')
b = find_encoding(BASE_DIR/ '2026/library_cafe/POS_02/register_sales.log')
c = find_encoding(BASE_DIR/ '2026/library_cafe/POS_03/register_sales.log')
d = find_encoding(BASE_DIR/ '2026/science_building/POS_02/register_sales.log')
print(a, b, c, d)



utf-8 iso-8859-1 utf-8 utf-8


In [15]:


def log_parser(file: Path)->list[dict]:
    """ Takes registerlogs as input, returns a list where each element is a dictionary of successfull transactions. """
    valid_lines = []
    transactions = [] 
    #opening file with correct input
    with open(file, encoding=find_encoding(file)) as f:
        line = f.readlines() #readlines create a list,line where each line in f becomes a object in that list.

    #step 1: removing all noise lines.
    for i in line:
        if i[0:2] =='[2':
            valid_lines.append(i) #if the object of line begins with [2, the first to chars of the time stamp of valid lines
    
    #step 2: Creating dictionaries for all valid lines, append them to a list of dictionaries    
    for valid_line in valid_lines:
        elements = valid_line.split()
        
        transaction = {
            'timestamp': elements[0].strip('[') +' '+ elements[1].strip(']'),
            'register' : elements[2].strip('[]'), 
            'item' : elements[3].split('=')[1],
            'price' : float(elements[4].split('=')[1]),
            'currency' : elements[5].strip(),
            'status' : elements[6].strip('()')
        }
        transactions.append(transaction)
    
    return transactions #return only the three first for visualiaztion



def valid_transactions(liste: list[dict]) -> list[dict]:
    """Filtering out only the valid transactions from log_parser"""
    return [transaction for transaction in liste if transaction['status'] == 'SUCCESS']



valid_transactions(log_parser('2026/library_cafe/POS_01/register_sales.log'))



[{'timestamp': '2026-10-15 07:00:21',
  'register': 'POS_01',
  'item': 'Espresso',
  'price': 2.66,
  'currency': 'NOK',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:01:57',
  'register': 'POS_01',
  'item': 'Croissant',
  'price': 4.04,
  'currency': 'EUR',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:02:31',
  'register': 'POS_01',
  'item': 'Sandwich',
  'price': 7.77,
  'currency': 'EUR',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:03:25',
  'register': 'POS_01',
  'item': 'Sandwich',
  'price': 7.19,
  'currency': 'USD',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:03:37',
  'register': 'POS_01',
  'item': 'Møffin',
  'price': 3.31,
  'currency': 'EUR',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:04:22',
  'register': 'POS_01',
  'item': 'Sandwich',
  'price': 7.09,
  'currency': 'EUR',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:10:01',
  'register': 'POS_01',
  'item': 'Latte',
  'price': 4.15,
  'currency': 'USD',
  'sta

Recap task 1: 
We see that using only Path and regular expressions we've created a set of functions that recieve raw log data from cashregisters, filters out noise lines and then sorts out only the successfull transactions. These transactions are then (as individual dictionaries) added to a list. The filtering for sorting out the lines that begin with '[2' and '[D', is maybe a bit primitive, but it will work until th timstamp hits th year 3000.

TASK 2

In [16]:



def dir_search(dirpath: str) -> list[str]:
    """Search through a directory. Add all paths of '.log' - files to a list. """
    log_files = []
    dirpath = Path(dirpath)
    # 1. searching through input directory
    for file in dirpath.iterdir():
        if file.is_file() and file.suffix == '.log':
            log_files.append(file)
      
        elif file.is_dir():
            #collects the logfiles if they are in subdirectories
            #if log_files list is local to the func we must do it this way in order to not clear out list in every recursive step
            log_files.extend(dir_search(file))
        
    return log_files


def find_active(data: Any) -> set[str]:
    """Returns at set of strings containing register name and whether it is active or not"""
    active_ids: set[str] = set()

    if isinstance(data, dict):
        # Check if current dir is active register
        if data.get('active') is True and 'id' in data:
            active_ids.add(str(data['id']))
        for value in data.values():
            active_ids.update(find_active(value))

    elif isinstance(data, list):
        #Check each list
        for item in data:
            active_ids.update(find_active(item))

    return active_ids




with open('store_config.json', 'r') as f:
    config_data = json.load(f)

find_active(config_data)

print(find_active(config_data),dir_search('2026')  )

dir_search('2026')




{'POS_03', 'POS_01', 'POS_02'} [WindowsPath('2026/library_cafe/POS_01/register_sales.log'), WindowsPath('2026/library_cafe/POS_02/register_sales.log'), WindowsPath('2026/library_cafe/POS_03/register_sales.log'), WindowsPath('2026/science_building/POS_01/register_sales.log'), WindowsPath('2026/science_building/POS_02/register_sales.log'), WindowsPath('2026/science_building/POS_03/register_sales.log'), WindowsPath('2026/student_union/POS_01/register_sales.log'), WindowsPath('2026/student_union/POS_02/register_sales.log'), WindowsPath('2026/student_union/POS_03/register_sales.log')]


[WindowsPath('2026/library_cafe/POS_01/register_sales.log'),
 WindowsPath('2026/library_cafe/POS_02/register_sales.log'),
 WindowsPath('2026/library_cafe/POS_03/register_sales.log'),
 WindowsPath('2026/science_building/POS_01/register_sales.log'),
 WindowsPath('2026/science_building/POS_02/register_sales.log'),
 WindowsPath('2026/science_building/POS_03/register_sales.log'),
 WindowsPath('2026/student_union/POS_01/register_sales.log'),
 WindowsPath('2026/student_union/POS_02/register_sales.log'),
 WindowsPath('2026/student_union/POS_03/register_sales.log')]

We see our two recursive funtions as the task asked for. The firs, dir_search takes a directory and searches through all subfiles and sub directories. If it finds a file with suffix .log it will append it to a list of strings containing filepaths. If it doesnt find a .log file it will call itself in a recursive step and start over in the next layer of subdirectories. The second function uses some of the same logic, where it searches for a special instance, and if it cant find it, it calls upon itself again to enter next layer. The difference is that this one searches lists and dicts.



TASK 3

In [17]:
"""TASK3.1"""


def mean(array: np.array) -> float:
    """Returns mean value of numpy array"""
    return float(sum(array)/len(array))


def stdev(array: np.array) -> float:
    """Returns standard deviation of array"""
    x_mean = mean(array)
    variance = sum((x-x_mean)**2 for x in array) /len(array)
    return float(m.sqrt(variance))


def valid_prices(dir) -> np.ndarray:
    """Takes a directory as input, returns prices from all transactions"""
    prices=[]
    filepaths = dir_search(dir)
    print(filepaths)

    for file_path in filepaths:
        transactions = log_parser(file_path)
        for trans in transactions:
            if 'price' in trans:
                prices.append(trans['price'])
    price_array = np.array(prices, dtype=float).copy()
    if price_array.size == 0:
        return price_array

    x_mean = mean(price_array)
    x_stdev = stdev(price_array)
    #create boolean values for if price is an outlier or not.
    valid_price_filter = np.abs(price_array - x_mean) <= 3*x_stdev

    clean_prices = price_array[valid_price_filter]
    return clean_prices


    
    
a = valid_prices('2026')
print(a)






[WindowsPath('2026/library_cafe/POS_01/register_sales.log'), WindowsPath('2026/library_cafe/POS_02/register_sales.log'), WindowsPath('2026/library_cafe/POS_03/register_sales.log'), WindowsPath('2026/science_building/POS_01/register_sales.log'), WindowsPath('2026/science_building/POS_02/register_sales.log'), WindowsPath('2026/science_building/POS_03/register_sales.log'), WindowsPath('2026/student_union/POS_01/register_sales.log'), WindowsPath('2026/student_union/POS_02/register_sales.log'), WindowsPath('2026/student_union/POS_03/register_sales.log')]
[2.66 3.57 4.04 7.77 7.19 3.31 7.09 4.54 4.43 4.15 7.54 7.29 7.01 4.3
 4.35 3.09 3.59 7.79 4.69 3.69 7.09 4.03 3.51 4.83 2.59 4.16 3.48 3.73
 3.11 4.16 3.7  4.09 4.21 7.36 3.19 3.54 4.02 3.62 3.06 3.49 4.76 3.48
 7.45 3.38 3.4  4.8  3.92 3.83 3.39 4.21 7.28 7.89 4.33 4.35 3.72 4.34
 4.08 4.1  7.49 4.18 4.55 4.6  2.94 4.17 3.67 4.17 4.46 3.3  4.37 7.63
 2.9  3.33 2.78 3.42 4.49 3.97 4.98 3.88 4.53 3.62 3.49 3.78 3.   7.45
 3.41 2.66 3.92 4.0

For task 3.1 to work properly we hasd to create to standard statistics functions to call up in the main filtering function. These two funcs simply calculate the mean and standard deviation of input arrays and return float values to be easily used later on. 
In the valid_prices function we use multiple previous functions to go throug our directory after .log files, prase them so that we get raw price data, to finally filter out the outlier values based on requirements in the task.

In [18]:
"""TASK 3.2"""

def get_items(filepaths):
        """Creates a list of the items in registerlogs"""
        items_set = set()

        for file_path in filepaths:
            transactions = log_parser(file_path)
            for transaction in transactions:
                if 'item' in transaction:
                    items_set.add(transaction['item'])
        return sorted(list(items_set))

def sale_refund_arrays(filepaths) -> np.ndarray:
    """Counts the sales and refunds, returns to arrays"""
    items = get_items(filepaths)
    print(f'Found items_ {items}')

    sales_dict = {item: 0 for item in items}
    refund_dict = {item: 0 for item in items}

    for file_path in filepaths:
         transactions = log_parser(file_path)
         for transaction in transactions:
              item = transaction.get('item')
              status = transaction.get('status')

              if item in sales_dict:
                    if status == 'SUCCESS':
                        sales_dict[item] += 1
                    elif status == 'REFUND':
                        refund_dict[item] += 1
    sales_counter = np.array([sales_dict[item] for item in items], dtype = float)
    refund_counter = np.array([refund_dict[item] for item in items], dtype=float)
    

    return sales_counter, refund_counter
    



def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Computes refund ratio based on count8 of refund and successes"""

    #add a base value in the denominator to avoid divide by zero errors
    base = 1e-6

    total_sales = sum(sales_count)
    total_refunds = sum(refund_count)
    total_ratio = total_refunds/ (total_sales + base)

    return f""" Refund ratio per item in found order = {refund_count/(sales_count + base)}
    Total Refund/Sales ratio = {total_ratio}
    )"""


ratios = compute_refund_ratio(*sale_refund_arrays(dir_search('2026')))
print(ratios)



Found items_ ['Cappuccino', 'Croissant', 'Espresso', 'Latte', 'Møffin', 'Sandwich']
 Refund ratio per item in found order = [0.1744186  0.2125     0.21917808 0.25925926 0.35937499 0.22093023]
    Total Refund/Sales ratio = 0.23617021226346763
    )


Here we use two 'help'-functions to reach our goal. The first returns to us each element that are sold in the cafès, using our previus log_parser. THe second counts all instances of the elements based on status, and the main takes the arrays from the second to compute all ratios with respect to items. And then we finally see the totalt refund ratio, even though, per item is the most interesting.

In [19]:


# task 4
def create_transaction_dataframe(transactions: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(transactions)
    if not df.empty and 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.set_index('timestamp')
        df = df.sort_index()
        return df

def resample_hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    return df.groupby('item').resample('1h')['price'].sum().unstack(level=0, fill_value=0)

def status_revenue_pivot(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    return df.pivot_table(index='item', columns='status', values='price', aggfunc='sum', fill_value=0)

def calculate_item_refund_ratios(df: pd.DataFrame) -> pd.Series:
    if df.empty:
        return pd.Series()
    
    success_counts = df[df['status'] == 'SUCCESS'].groupby('item')['price'].count()
    refund_counts = df[df['status'] == 'REFUND'].groupby('item')['price'].count()
    summary_df = pd.DataFrame({'sales': success_counts, 'refunds': refund_counts}).fillna(0) # Riktig innrykk
    
    base = 1e-6
    ratios = summary_df['refunds'] / (summary_df['sales'] + base)
    return ratios


# task 5


def run_coffee_shop_pipeline(root_dir: str, config_path: str):
    log_files = dir_search(root_dir)
    
    with open(config_path, 'r', encoding='utf-8') as f:
        config_data = json.load(f)
    active_registers = find_active(config_data)
    
    all_transactions = []
    for file_path in log_files:
        parsed_items = log_parser(file_path)
        for trans in parsed_items:
            if trans.get('register') in active_registers:
                all_transactions.append(trans)
                
    df = create_transaction_dataframe(all_transactions)
    
    if not df.empty:
        
        price_array = df['price'].to_numpy(dtype=float)
        x_mean = np.mean(price_array)
        x_stdev = np.std(price_array)
        median_val = np.median(price_array[np.abs(price_array - x_mean) <= 3 * x_stdev])
        
        outlier_mask = np.abs(price_array - x_mean) > 3 * x_stdev
        price_array = np.where(outlier_mask, median_val, price_array)
        df['price'] = price_array
        
        df.loc[df['currency'] == 'NOK', 'price'] *= 10.0
      
        hourly_rev = resample_hourly_revenue(df)
        pivot_table = status_revenue_pivot(df)
        refund_ratios = calculate_item_refund_ratios(df)
        
        df.reset_index().to_csv('cleaned_transactions.csv', index=False)
        hourly_rev.to_json('hourly_revenue.json', orient='index')
        pivot_table.to_csv('status_pivot_table.csv')
        refund_ratios.to_csv('item_refund_ratios.csv', header=['refund_ratio'])
        
        print("Arbeidsflyten er fullført! Alllefiler er generert.")

run_coffee_shop_pipeline('2026', 'store_config.json')

Arbeidsflyten er fullført! Alllefiler er generert.


C:\Users\danie\AppData\Local\Temp\ipykernel_30884\218129981.py:70: Pandas4Warning: The default formatting of datetime/timedelta values will change from numbers ("epoch") to strings ("iso") in a future version. Specify `date_format="iso"` explicitly in the `to_json()` call to opt-in to the future behaviour and silence this warning.
  hourly_rev.to_json('hourly_revenue.json', orient='index')


Task 4: Functions that give key numbers about the cafès, they takes lists and returns pandas DF's or take Df's as input and manipulate them into presenting relevant data.

Task 5: Streamlines all funtions above so that the user can give directory and config_file as input and in return get the key numbers as csv and json files.

In [20]:
import ensurepip
ensurepip.bootstrap()



In [21]:
!python -m pip install pandas numpy


In [22]:
"""Printer ut hvor de genererte filnene blir lagt"""
import os
print("Nåværende mappe:", os.getcwd())

Nåværende mappe: c:\Users\danie\Documents\NMBU 4.år\Inf 201\Deliverables\W43_44\INF201_Deliverable_2


In [23]:
import sys
import subprocess
subprocess.run([sys.executable, "generate_data_for_d2 (1).py"])

CompletedProcess(args=['c:\\Users\\danie\\Documents\\NMBU 4.år\\Inf 201\\Deliverables\\W43_44\\INF201_Deliverable_2\\.venv\\Scripts\\python.exe', 'generate_data_for_d2 (1).py'], returncode=2)